# Figures of the article

Writes the pgfplots source of every data figure of the article and of the Supplementary material from the results of the other notebooks (`../results` and `../figdata`), and compiles every source in `../figures` with pdflatex. Each figure is a single standalone file that uses the common style `figstyle.tex`, written by this notebook; its data are written next to it in `../figures/data`, so that the sources can be edited and recompiled without Python. The figures are drawn at the text width of the article, 170 mm, with the same fonts as the manuscript.

In [ ]:
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RESULTS = ROOT / "results"
FIGDATA = ROOT / "figdata"
LITERATURE = ROOT / "src" / "ccpyro" / "data"
FIGURES = ROOT / "figures"
DATA = FIGURES / "data"
DATA.mkdir(parents=True, exist_ok=True)

FEEDSTOCKS = [
    ("huskp", "rice husk (pelletized)", "rh", "RH"),
    ("blendp", "blend (pelletized)", "bl", "RH:CC"),
    ("cobp", "corncob (pelletized)", "cc", "CC"),
]
yields = pd.read_csv(FIGDATA / "yields.csv")
calibration = pd.read_csv(RESULTS / "model_calibration.csv").set_index("treatment")
print(len(yields), "nodes;", ", ".join(code for code, *_ in FEEDSTOCKS))

## Common pieces

Every panel uses the style `cce` of `figstyle.tex`. The three feedstocks keep the same colour, dash pattern and marker in every figure (styles `rh`, `bl` and `cc`), and each multi-panel figure carries a single legend above the panels.

In [ ]:
FIGSTYLE = r"""\usepackage{newtxtext}
\usepackage{newtxmath}
\usepackage{amsmath}
\usepackage{siunitx}
\usepackage{chemformula}
\usepackage{tikz}
\usepackage{pgfplots}
\usepgfplotslibrary{groupplots, fillbetween}
\usetikzlibrary{calc}

\pgfplotsset{compat=1.18, table/col sep=comma}
\sisetup{detect-all, per-mode=symbol}

\definecolor{rhcol}{HTML}{0072B2}
\definecolor{blcol}{HTML}{E69F00}
\definecolor{cccol}{HTML}{009E73}
\definecolor{greycol}{HTML}{6E6E6E}

\pgfplotsset{
	cce/.style={
		scale only axis,
		axis lines*=left,
		axis line style={line width=0.5pt, black!85},
		every tick/.style={black!85, line width=0.4pt},
		tick align=outside,
		major tick length=2.2pt,
		ymajorgrids,
		grid style={line width=0.25pt, black!12},
		label style={font=\footnotesize},
		tick label style={font=\scriptsize},
		title style={font=\footnotesize\bfseries, yshift=-0.5mm},
		legend style={font=\footnotesize, draw=none, fill=none,
		              /tikz/every even column/.append style={column sep=4mm}},
		legend cell align=left,
		scaled y ticks=false,
		yticklabel style={/pgf/number format/fixed, /pgf/number format/precision=2},
		xticklabel style={/pgf/number format/fixed},
		clip mode=individual,
		error bars/error bar style={line width=0.5pt, solid},
		error bars/error mark options={rotate=90, mark size=1.4pt, line width=0.5pt, solid},
	},
	rh/.style={rhcol, line width=0.9pt, mark=*, mark size=1.5pt, mark options={solid}},
	bl/.style={blcol, line width=0.9pt, dashed, mark=triangle*, mark size=1.9pt, mark options={solid}},
	cc/.style={cccol, line width=0.9pt, dash dot, mark=square*, mark size=1.3pt, mark options={solid}},
	band/.style={draw=none, fill opacity=0.16},
	creck/.style={densely dotted, line width=1.0pt, no marks},
	prediction/.style={greycol, only marks, mark=triangle*, mark size=2.1pt, mark options={fill=white, line width=0.6pt}},
	literature/.style={greycol, only marks, mark=o, mark size=1.5pt, mark options={line width=0.5pt}},
	literature tar/.style={greycol, only marks, mark=x, mark size=2.2pt, mark options={line width=0.7pt}},
	estimate/.style={black, line width=0.7pt, mark=*, mark size=1.1pt},
	level1/.style={draw=none, mark=none, forget plot, error bars/y dir=both, error bars/y explicit,
	               error bars/error mark=none, error bars/error bar style={line width=1.3pt}},
	level2/.style={level1, error bars/error bar style={line width=2.6pt}},
	level3/.style={level1, error bars/error bar style={line width=4.2pt}},
	interval/.style={legend image code/.code={\draw[##1] (0.15cm, -0.13cm) -- (0.15cm, 0.13cm);}},
}
\tikzset{panel label/.style={font=\footnotesize\bfseries, anchor=south west, inner sep=0pt, yshift=1.8mm}}
"""
(FIGURES / "figstyle.tex").write_text(FIGSTYLE, encoding="utf-8")

In [ ]:
TEMPERATURE = r"Temperature (\unit{\degreeCelsius})"
KGKG = r"(\unit{\kilo\gram\per\kilo\gram})"
MOLMOL = r"(\unit{\mole\per\mole})"


def save(name, frame):
    """Write the data of a figure next to its source and return the relative path used by pgfplots."""
    frame.to_csv(DATA / f"{name}.csv", index=False, float_format="%.6g")
    return f"data/{name}.csv"


def document(body, tables=()):
    reads = "".join(f"\\pgfplotstableread{{{path}}}{macro}\n" for macro, path in tables)
    return ("\\documentclass[border=1pt, tikz]{standalone}\n\\input{figstyle}\n\n\\begin{document}\n" + reads
            + "\\begin{tikzpicture}\n" + body + "\\end{tikzpicture}\n\\end{document}\n")


def write(name, text):
    (FIGURES / f"{name}.tex").write_text(text, encoding="utf-8")
    print("wrote", name)


def legend_entries(extra=()):
    """Legend images of the three feedstocks, followed by any extra (image, label) pair."""
    lines = [f"\t\t\\addlegendimage{{{style}}}\n\t\t\\addlegendentry{{{label}}}\n"
             for _, _, style, label in FEEDSTOCKS]
    lines += [f"\t\t\\addlegendimage{{{image}}}\n\t\t\\addlegendentry{{{label}}}\n" for image, label in extra]
    return "".join(lines)


def legend_above(name, first, last, lift="7mm"):
    return (f"\t\\node[anchor=south] at ($({first}.north west)!0.5!({last}.north east) + (0, {lift})$) "
            f"{{\\pgfplotslegendfromname{{{name}}}}};\n")


def panel(letter):
    return f"\t\t\\node[panel label] at (rel axis cs:0, 1) {{({letter})}};\n"


def band(table, code, column, style):
    """Shaded band of plus and minus one Monte Carlo standard deviation."""
    return (f"\t\t\\addplot[name path=up, draw=none, forget plot] table[x=T, y={column}_hi_{code}] {{{table}}};\n"
            f"\t\t\\addplot[name path=lo, draw=none, forget plot] table[x=T, y={column}_lo_{code}] {{{table}}};\n"
            f"\t\t\\addplot[band, {style}, forget plot] fill between[of=up and lo];\n")


def with_bands(columns, clip_low=0.0):
    """Node table with the lower and upper ends of the band of each column."""
    frame = yields[["T"]].copy()
    for code, *_ in FEEDSTOCKS:
        for column in columns:
            value, sd = yields[f"{column}_{code}"], yields[f"{column}_sd_{code}"]
            frame[f"{column}_{code}"] = value
            frame[f"{column}_lo_{code}"] = np.maximum(value - sd, clip_low)
            frame[f"{column}_hi_{code}"] = value + sd
            frame[f"{column}_sd_{code}"] = sd
    return frame

## Figure 3. Thermal basis

In [ ]:
def figure_3():
    nodes = save("fig3_nodes", with_bands(["w_char"]))
    curves = {}
    for code, *_ in FEEDSTOCKS:
        curve = pd.read_csv(FIGDATA / f"thermal_{code}.csv")
        curve["dtg"] = 100 * curve["rate"]
        curves[code] = save(f"fig3_thermal_{code}", curve[["T", "theta", "dtg"]])
    body = ("\t\\begin{groupplot}[cce, group style={group size=2 by 1, horizontal sep=17mm},\n"
            "\t\twidth=70mm, height=48mm, xmin=200, xmax=850, xtick={200,300,...,800}, xlabel={%s}]\n" % TEMPERATURE)
    body += (f"\t\\nextgroupplot[ylabel={{Solid yield $\\theta$ {KGKG}}}, ymin=0, ymax=1.05,\n"
             "\t\tlegend to name=leg3, legend style={legend columns=-1}]\n")
    body += legend_entries()
    for code, _, style, _ in FEEDSTOCKS:
        body += f"\t\t\\addplot[{style}, no marks] table[x=T, y=theta] {{{curves[code]}}};\n"
        body += (f"\t\t\\addplot[{style}, only marks, error bars/y dir=both, error bars/y explicit]\n"
                 f"\t\t\ttable[x=T, y=w_char_{code}, y error=w_char_sd_{code}] {{{nodes}}};\n")
    body += panel("a")
    body += f"\t\\nextgroupplot[ylabel={{DTG (\\unit{{\\percent\\per\\degreeCelsius}})}}, ymin=0, ymax=1.0]\n"
    for code, treatment, style, _ in FEEDSTOCKS:
        curve = pd.read_csv(FIGDATA / f"thermal_{code}.csv")
        peak_T = float(calibration.loc[treatment, "T_M"])
        peak = float(np.interp(peak_T, curve["T"], 100 * curve["rate"]))
        body += f"\t\t\\addplot[{style}, no marks] table[x=T, y=dtg] {{{curves[code]}}};\n"
        body += (f"\t\t\\draw[{style}col, line width=0.6pt, densely dotted] (axis cs:{peak_T:.1f}, 0) -- "
                 f"(axis cs:{peak_T:.1f}, {peak:.3f});\n")
    body += panel("b") + "\t\\end{groupplot}\n" + legend_above("leg3", "group c1r1", "group c2r1")
    write("Figure_3", document(body))

## Figure 4. Product streams

In [ ]:
def two_by_two(name, panels, legend_name, extra_legend=(), first_options=""):
    """Four panels sharing the temperature axis; each panel is (letter, ylabel, ymin, ymax, content)."""
    body = ("\t\\begin{groupplot}[cce, group style={group size=2 by 2, horizontal sep=17mm, vertical sep=9mm,\n"
            "\t\txlabels at=edge bottom, xticklabels at=edge bottom},\n"
            "\t\twidth=70mm, height=42mm, xmin=250, xmax=850, xtick={300,400,...,800}, xlabel={%s}]\n" % TEMPERATURE)
    for index, (letter, ylabel, ymin, ymax, content) in enumerate(panels):
        options = f"ylabel={{{ylabel}}}, ymin={ymin}, ymax={ymax}"
        if index == 0:
            options += f", legend to name={legend_name}, legend style={{legend columns=-1}}{first_options}"
        body += f"\t\\nextgroupplot[{options}]\n"
        if index == 0:
            body += legend_entries(extra_legend)
        body += content + panel(letter)
    body += "\t\\end{groupplot}\n" + legend_above(legend_name, "group c1r1", "group c2r1")
    write(name, document(body))


def stream_content(table, column):
    content = ""
    for code, _, style, _ in FEEDSTOCKS:
        content += band(table, code, column, f"fill={style}col")
        content += f"\t\t\\addplot[{style}] table[x=T, y={column}_{code}] {{{table}}};\n"
    return content


def figure_4():
    table = save("fig4_streams", with_bands(["w_tar", "w_water", "w_gas", "water_in_liquid"]))
    two_by_two("Figure_4", [
        ("a", rf"$w_{{\mathrm{{tar}}}}$ {KGKG}", 0, 0.30, stream_content(table, "w_tar")),
        ("b", rf"$w_{{\ch{{H2O}}}}$ {KGKG}", 0, 0.25, stream_content(table, "w_water")),
        ("c", rf"$w_{{\mathrm{{gas}}}}$ {KGKG}", 0, 0.75, stream_content(table, "w_gas")),
        ("d", rf"$\omega_{{\mathrm{{W}}}}$ {KGKG}", 0, 1.05, stream_content(table, "water_in_liquid")),
    ], "leg4")

## Figure 5. Released gas and the additive prediction of the blend

In [ ]:
def figure_5():
    additivity = pd.read_csv(RESULTS / "gas_additivity.csv")
    frame = yields[["T"]].copy()
    for species in ("CO", "CO2", "CH4", "H2"):
        for code, *_ in FEEDSTOCKS:
            for kind in ("model", "exp", "sd"):
                frame[f"{species}_{kind}_{code}"] = yields[f"phi_{species}_{kind}_{code}"]
        predicted = additivity[additivity.species == species].set_index("T")["predicted_yield_mixing"]
        frame[f"{species}_pred"] = frame["T"].map(predicted)
    table = save("fig5_gas", frame)
    limits = {"CO": 0.45, "CO2": 0.65, "CH4": 0.45, "H2": 0.55}
    panels = []
    for letter, species in zip("abcd", ("CO", "CO2", "CH4", "H2")):
        content = ""
        for code, _, style, _ in FEEDSTOCKS:
            content += f"\t\t\\addplot[{style}, no marks] table[x=T, y={species}_model_{code}] {{{table}}};\n"
            content += (f"\t\t\\addplot[{style}, only marks, error bars/y dir=both, error bars/y explicit]\n"
                        f"\t\t\ttable[x=T, y={species}_exp_{code}, y error={species}_sd_{code}] {{{table}}};\n")
        content += f"\t\t\\addplot[prediction] table[x=T, y={species}_pred] {{{table}}};\n"
        panels.append((letter, rf"$\varphi_{{\ch{{{species}}}}}$ {MOLMOL}", 0, limits[species], content))
    two_by_two("Figure_5", panels, "leg5", extra_legend=[("prediction", "RH:CC, additive prediction")])

## Figure 6. Van Krevelen trajectories of the char and the tar

In [ ]:
def figure_6(split=600.0):
    frame = yields[["T"]].copy()
    for code, *_ in FEEDSTOCKS:
        for phase in ("c", "t"):
            for axis in ("x", "y"):
                column = f"{axis}_{phase}_{code}"
                frame[f"{column}_det"] = yields[column].where(yields["T"] <= split)
                frame[f"{column}_reg"] = yields[column].where(yields["T"] >= split)
    table = save("fig6_vk", frame)
    char = save("fig6_literature_char", pd.read_csv(LITERATURE / "literature_char.csv"))
    tar = save("fig6_literature_tar", pd.read_csv(LITERATURE / "literature_tar.csv"))
    body = ("\t\\begin{groupplot}[cce, group style={group size=2 by 1, horizontal sep=17mm},\n"
            "\t\twidth=70mm, height=52mm, xmin=0, xmax=0.8, xtick={0,0.2,...,0.8}, ymin=0, ymax=1.6,\n"
            f"\t\txlabel={{O/C {MOLMOL}}}, ylabel={{H/C {MOLMOL}}}, unbounded coords=jump, xmajorgrids]\n")
    for index, (letter, phase, literature, title) in enumerate((("a", "c", char, "Char"), ("b", "t", tar, "Tar"))):
        options = "title={%s}" % title
        if index == 0:
            options += ", legend to name=leg6, legend style={legend columns=-1}"
        body += f"\t\\nextgroupplot[{options}]\n"
        if index == 0:
            body += legend_entries([("literature", "Published analyses"),
                                    ("black, thick, densely dotted", r"Above \qty{600}{\degreeCelsius}")])
        body += f"\t\t\\addplot[literature] table[x=OC, y=HC] {{{literature}}};\n"
        for code, _, style, _ in FEEDSTOCKS:
            body += f"\t\t\\addplot[{style}] table[x=y_{phase}_{code}_det, y=x_{phase}_{code}_det] {{{table}}};\n"
            body += (f"\t\t\\addplot[{style}, densely dotted, mark options={{solid}}]\n"
                     f"\t\t\ttable[x=y_{phase}_{code}_reg, y=x_{phase}_{code}_reg] {{{table}}};\n")
        body += panel(letter)
    body += "\t\\end{groupplot}\n" + legend_above("leg6", "group c1r1", "group c2r1", "11mm")
    write("Figure_6", document(body))

## Figure 7. Profile-likelihood intervals of the tar composition

In [ ]:
def figure_7():
    levels = (("measurements", "level1"), ("closures", "level2"), ("estimator", "level3"))
    body = ("\t\\begin{groupplot}[cce, group style={group size=3 by 2, horizontal sep=6mm, vertical sep=9mm,\n"
            "\t\txlabels at=edge bottom, xticklabels at=edge bottom, ylabels at=edge left,\n"
            "\t\tyticklabels at=edge left},\n"
            "\t\twidth=49mm, height=40mm, xmin=250, xmax=850, xtick={300,400,...,800},\n"
            f"\t\txlabel={{{TEMPERATURE}}}, xmajorgrids]\n")
    letter = iter("abcdef")
    for row, (variable, ymax, label) in enumerate((("y_t", 0.85, r"$y_{\mathrm{t}}$"), ("x_t", 1.65, r"$x_{\mathrm{t}}$"))):
        table = save(f"fig7_{variable}", pd.read_csv(FIGDATA / f"profile_{variable}.csv"))
        for column, (code, _, style, title) in enumerate(FEEDSTOCKS):
            options = [f"ymin=0, ymax={ymax}"] + ([f"ylabel={{{label} {MOLMOL}}}"] if column == 0 else [])
            if row == 0:
                options.append(f"title={{{title}}}")
            if row == 0 and column == 0:
                options.append("legend to name=leg7, legend style={legend columns=-1}")
            body += f"\t\\nextgroupplot[{', '.join(options)}]\n"
            if row == 0 and column == 0:
                for image, text in (("interval, black!40, line width=1.3pt", "Measurements"),
                                    ("interval, black!60, line width=2.6pt", r"$+$ closures"),
                                    ("interval, black, line width=4.2pt", r"$+$ regularisation")):
                    body += f"\t\t\\addlegendimage{{{image}}}\n\t\t\\addlegendentry{{{text}}}\n"
                body += "\t\t\\addlegendimage{estimate}\n\t\t\\addlegendentry{Point estimate}\n"
            colours = {"level1": "black!40", "level2": f"{style}col!50", "level3": f"{style}col"}
            for level, level_style in levels:
                body += (f"\t\t\\addplot[{level_style}, {colours[level_style]}]\n"
                         f"\t\t\ttable[x=T, y expr=0.5*(\\thisrow{{{level}_lo_{code}}}+\\thisrow{{{level}_hi_{code}}}),\n"
                         f"\t\t\ty error expr=0.5*(\\thisrow{{{level}_hi_{code}}}-\\thisrow{{{level}_lo_{code}}})] {{{table}}};\n")
            body += f"\t\t\\addplot[estimate] table[x=T, y=best_{code}] {{{table}}};\n"
            body += panel(next(letter))
    body += "\t\\end{groupplot}\n" + legend_above("leg7", "group c1r1", "group c3r1", "11mm")
    write("Figure_7", document(body))

## Figure 8. CRECK mechanism against the measurements and the framework

In [ ]:
def figure_8():
    curves = save("fig8_creck_curves", pd.read_csv(FIGDATA / "creck_curves.csv"))
    stage = save("fig8_creck_gas", pd.read_csv(FIGDATA / "creck_stage_gas.csv"))
    nodes = save("fig8_nodes", with_bands(["w_tar", "y_t"]).join(
        yields[[f"phi_CH4_exp_{code}" for code, *_ in FEEDSTOCKS]]))
    tar = save("fig8_literature_tar", pd.read_csv(LITERATURE / "literature_tar.csv"))
    thermal = {code: f"data/fig3_thermal_{code}.csv" for code, *_ in FEEDSTOCKS}
    a = b = c = d = ""
    for code, _, style, _ in FEEDSTOCKS:
        a += f"\t\t\\addplot[{style}, no marks] table[x=T, y=theta] {{{thermal[code]}}};\n"
        a += f"\t\t\\addplot[{style}, creck] table[x=T, y=theta_{code}] {{{curves}}};\n"
        b += (f"\t\t\\addplot[{style}, only marks, error bars/y dir=both, error bars/y explicit]\n"
              f"\t\t\ttable[x=T, y=w_tar_{code}, y error=w_tar_sd_{code}] {{{nodes}}};\n")
        b += f"\t\t\\addplot[{style}, creck] table[x=T, y=w_tar_{code}] {{{curves}}};\n"
        c += (f"\t\t\\addplot[{style}, only marks, error bars/y dir=both, error bars/y explicit]\n"
              f"\t\t\ttable[x=T, y=y_t_{code}, y error=y_t_sd_{code}] {{{nodes}}};\n")
        c += f"\t\t\\addplot[{style}, creck] table[x=T, y=y_t_{code}] {{{curves}}};\n"
        d += f"\t\t\\addplot[{style}, only marks] table[x=T, y=phi_CH4_exp_{code}] {{{nodes}}};\n"
        d += (f"\t\t\\addplot[{style}, densely dotted, line width=1.0pt, mark options={{fill=white, solid}}]\n"
              f"\t\t\ttable[x=T, y=phi_CH4_{code}] {{{stage}}};\n")
    c = f"\t\t\\addplot[literature tar] table[x=T, y=OC] {{{tar}}};\n" + c
    body = ("\t\\begin{groupplot}[cce, group style={group size=2 by 2, horizontal sep=17mm, vertical sep=9mm,\n"
            "\t\txlabels at=edge bottom, xticklabels at=edge bottom},\n"
            "\t\twidth=70mm, height=42mm, xmin=200, xmax=850, xtick={200,300,...,800}, xlabel={%s}]\n" % TEMPERATURE)
    panels = [("a", rf"Solid yield $\theta$ {KGKG}", 1.05, a), ("b", rf"$w_{{\mathrm{{tar}}}}$ {KGKG}", 0.65, b),
              ("c", rf"$y_{{\mathrm{{t}}}}$ {MOLMOL}", 0.8, c), ("d", rf"$\varphi_{{\ch{{CH4}}}}$ {MOLMOL}", 0.45, d)]
    for index, (letter, ylabel, ymax, content) in enumerate(panels):
        options = f"ylabel={{{ylabel}}}, ymin=0, ymax={ymax}"
        if index == 0:
            options += ", legend to name=leg8, legend style={legend columns=-1}"
        body += f"\t\\nextgroupplot[{options}]\n"
        if index == 0:
            body += legend_entries([("black, creck", "CRECK mechanism"), ("literature tar", "Published tars")])
        body += content + panel(letter)
    body += "\t\\end{groupplot}\n" + legend_above("leg8", "group c1r1", "group c2r1")
    write("Figure_8", document(body))

## Figure 9 and Figure S2. Energy indices and hydrogen yield

In [ ]:
def figure_9_s2():
    table = save("fig9_energy", with_bands(["LHV_gas", "CGE", "Y_H2"]))
    body = ("\t\\begin{groupplot}[cce, group style={group size=2 by 1, horizontal sep=17mm},\n"
            "\t\twidth=70mm, height=48mm, xmin=250, xmax=850, xtick={300,400,...,800}, xlabel={%s}]\n" % TEMPERATURE)
    body += ("\t\\nextgroupplot[ylabel={$LHV_{\\mathrm{gas}}$ (\\unit{\\mega\\joule\\per\\cubic\\metre})}, ymin=0, "
             "ymax=20, legend to name=leg9, legend style={legend columns=-1}]\n")
    body += legend_entries() + stream_content(table, "LHV_gas") + panel("a")
    body += "\t\\nextgroupplot[ylabel={$\\eta_{\\mathrm{CGE}}$ (\\unit{\\percent})}, ymin=0, ymax=60]\n"
    body += stream_content(table, "CGE") + panel("b")
    body += "\t\\end{groupplot}\n" + legend_above("leg9", "group c1r1", "group c2r1")
    write("Figure_9", document(body))
    body = ("\t\\begin{axis}[cce, width=70mm, height=48mm, xmin=250, xmax=850, xtick={300,400,...,800},\n"
            f"\t\txlabel={{{TEMPERATURE}}}, ylabel={{$Y_{{\\ch{{H2}}}}$ (\\unit{{\\milli\\litre\\per\\gram}})}},\n"
            "\t\tymin=0, ymax=160, legend pos=north west, legend style={legend columns=1}]\n")
    body += legend_entries() + stream_content(table, "Y_H2") + "\t\\end{axis}\n"
    write("Figure_S2", document(body))

## Writing and compiling every figure

The cell below writes the data figures and compiles their sources. The compilation needs a LaTeX distribution with pdflatex; without it, the sources are written and can be compiled later.

In [ ]:
figure_3()
figure_4()
figure_5()
figure_6()
figure_7()
figure_8()
figure_9_s2()

if shutil.which("pdflatex"):
    for source in sorted(FIGURES.glob("Figure_*.tex")):
        run = subprocess.run(["pdflatex", "-interaction=nonstopmode", "-halt-on-error", source.name],
                             cwd=FIGURES, capture_output=True, text=True)
        status = "ok" if run.returncode == 0 else "FAILED, see " + source.with_suffix(".log").name
        print(f"{source.stem:12s} {status}")
    for extension in ("aux", "log"):
        for leftover in FIGURES.glob(f"Figure_*.{extension}"):
            if (FIGURES / f"{leftover.stem}.pdf").exists():
                leftover.unlink()
else:
    print("pdflatex not found; the sources are in", FIGURES)

The data figures of the article are `Figure_3.pdf` to `Figure_9.pdf`, numbered as in the article; `Figure_S2.pdf` is the hydrogen yield of the Supplementary material.